<a href="https://colab.research.google.com/github/ChristopherMwanginjoroge/deep-learning/blob/main/imageclassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

transform=transforms.ToTensor()

train_data=datasets.FashionMNIST(root='data',train=True,download=True,transform=transform)
test_data=datasets.FashionMNIST(root='data',train=False,download=True,transform=transform)

print(test_data)

Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()


In [5]:
train_loader=DataLoader(train_data,batch_size=64,shuffle=True)
test_loader=DataLoader(test_data,batch_size=64,shuffle=True)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

images, labels = next(iter(train_loader))

print(images.shape)

torch.Size([64, 1, 28, 28])


In [6]:
class FashionCNN(nn.Module):
  def __init__(self):
    super(FashionCNN, self).__init__()

    self.conv1=nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,padding=1)
    self.pool1=nn.MaxPool2d(kernel_size=2,stride=2)

    self.conv2=nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1)
    self.pool2=nn.MaxPool2d(kernel_size=2,stride=2)

    self.fc1=nn.Linear(in_features=64*7*7,out_features=128)

    self.dropout=nn.Dropout(p=0.05)

    self.fc2 = nn.Linear(in_features=128, out_features=10)


  def forward(self,x):
    x=F.relu(self.conv1(x))
    x=self.pool1(x)

    x=F.relu(self.conv2(x))
    x=self.pool2(x)

    x=x.view(-1,64*7*7)
    x=F.relu(self.fc1(x))
    x=self.dropout(x)
    x=self.fc2(x)

    return x

model=FashionCNN()
print(model)

FashionCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (dropout): Dropout(p=0.05, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [9]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

epochs=10

for epoch in range(epochs):
  model.train()
  running_loss=0.0

  for image,label in train_loader:
    outputs=model(images)

    loss=criterion(outputs,labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    running_loss+=loss.item()

  if (epoch+1)%2==0:
    avg_loss=running_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")



Epoch [2/10], Loss: 0.0000
Epoch [4/10], Loss: 0.0000
Epoch [6/10], Loss: 0.0000
Epoch [8/10], Loss: 0.0193
Epoch [10/10], Loss: 0.0000


In [10]:
model.eval()
correct=0
total=0

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

with torch.no_grad():
  for images,labels in test_loader:
    outputs=model(images)

    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\nFinal CNN Test Accuracy on Fashion-MNIST: {accuracy:.2f}%")


Final CNN Test Accuracy on Fashion-MNIST: 69.45%
